## Curso Integral de Econometría 
### Econometría de datos panel con Python
### Tema: Estimación del modelo dinámico con una variable dependiente rezagada
### Subtema: El ingreso y la convergencia económica en América Latina
### Sesión: 14
### Fecha: 21/11/2026
### Asesor: Alexis Adonai Morales Alberto

#### **Configuración de la sesión: Módulos, submódulos, clases y directorios**

In [3]:
# Instalación previa del módulo pydynpd

!pip install pydynpd

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [pydynpd]
   -------------------- ------------------- 1/2 [pydynpd]
   ---------------------------------------- 2/2 [pydynpd]




[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Módulos 

import contextlib
import io
import json
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf


# Submódulos y/o clases 

from pathlib import Path
from scipy.stats import norm
from pydynpd import regression

In [3]:
BASE = Path(
    "G:\Econometrics Data Lab\Curso integral de econometría\Datos de panel\PANEL_SEP26"
)
if not (BASE / "Datos").exists() and (BASE / "S14_Panel_Dinamico").exists():
    BASE = BASE / "S14_Panel_Dinamico"
SALIDA = BASE / "Resultados"
SALIDA.mkdir(exist_ok=True)
MOSTRAR_GRAFICOS = True
ALPHA = .05
colores = ["#F04A1D", "#84517F", "#65773E", "#6C6259"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.facecolor": "white"})

#### **Modelo y decisiones previas**

In [4]:
X_economicas = ["inversion_fbkf", "apertura_comercial", "inflacion"]
variables = ["ln_pib_pc"] + X_economicas
ARCHIVO_DATOS = BASE / "Datos" / "Datos_AL_Panel.csv"
if not ARCHIVO_DATOS.is_file():
    alternativa = BASE / "Datos_AL_Panel.csv"
    if alternativa.is_file():
        ARCHIVO_DATOS = alternativa
    else:
        raise FileNotFoundError(
            f"No se encontró {ARCHIVO_DATOS}. Descomprime el paquete completo "
            "o coloca Datos_AL_Panel.csv junto al notebook."
        )
Panel = pd.read_csv(ARCHIVO_DATOS, encoding="utf-8-sig")
requeridas_base = ["pais", "anio"] + variables
faltan_columnas = sorted(set(requeridas_base) - set(Panel.columns))
if faltan_columnas:
    raise ValueError(f"Faltan columnas en el CSV: {faltan_columnas}")
Panel = Panel[requeridas_base].copy()
for v in ["anio"] + variables:
    Panel[v] = pd.to_numeric(Panel[v], errors="raise")
if Panel[requeridas_base].isna().any().any():
    raise ValueError("Hay valores faltantes en las variables requeridas del CSV.")
if not np.isfinite(Panel[["anio"] + variables].to_numpy(float)).all():
    raise ValueError("El CSV contiene valores numéricos infinitos.")
if not Panel.anio.mod(1).eq(0).all():
    raise ValueError("Los años deben ser enteros.")
Panel["anio"] = Panel["anio"].astype(int)
if Panel.duplicated(["pais", "anio"]).any():
    raise ValueError("Hay filas duplicadas de país y año.")
Panel = Panel.sort_values(["pais", "anio"]).reset_index(drop=True)
for pais, grupo in Panel.groupby("pais", sort=True):
    if not np.array_equal(grupo.anio.to_numpy(), np.arange(2011, 2023)):
        raise ValueError(f"{pais}: se esperan los años 2011–2022 completos; "
                         f"se encontraron {grupo.anio.tolist()}.")
print("Países:", sorted(Panel.pais.unique()), "| Base: 2011–2022")

Países: ['BRA', 'CHL', 'COL', 'CRI', 'ECU', 'MEX', 'PER', 'URY'] | Base: 2011–2022


#### **Rezagos por calendario**

In [5]:
p = Panel[["pais", "anio"] + variables].copy()
for k in [1, 2, 3]:
    rezago = p[["pais", "anio"] + variables].copy()
    rezago["anio"] = rezago["anio"] + k
    rezago = rezago.rename(columns={v: f"L{k}_{v}" for v in variables})
    p = p.merge(rezago, on=["pais", "anio"], how="left", validate="one_to_one")
for v in variables:
    p[f"d_{v}"] = p[v] - p[f"L1_{v}"]
p["L1_d_ln_pib_pc"] = p["L1_ln_pib_pc"] - p["L2_ln_pib_pc"]
p["shock_2020"] = p.anio.eq(2020).astype(float)
p["rebote_2021"] = p.anio.eq(2021).astype(float)
for v in ["shock_2020", "rebote_2021"]:
    rezago = p[["pais", "anio", v]].copy()
    rezago["anio"] += 1
    rezago = rezago.rename(columns={v: f"L1_{v}"})
    p = p.merge(rezago, on=["pais", "anio"], how="left", validate="one_to_one")
    p[f"d_{v}"] = p[v] - p[f"L1_{v}"]
xvars = ["L1_d_ln_pib_pc", "d_inversion_fbkf", "d_apertura_comercial",
         "d_inflacion", "d_shock_2020", "d_rebote_2021"]
zvars = ["L2_ln_pib_pc", "L3_ln_pib_pc", "L2_inversion_fbkf",
         "L2_apertura_comercial", "L2_inflacion", "d_shock_2020", "d_rebote_2021"]
d = p.loc[p.anio.between(2013, 2022)].sort_values(["pais", "anio"]).reset_index(drop=True).copy()
falta_L3 = d["L3_ln_pib_pc"].isna()
inesperados = falta_L3 & d.anio.ne(2013)
if inesperados.any():
    raise ValueError("Falta L3 fuera de 2013:\n" +
                     d.loc[inesperados, ["pais", "anio"]].to_string(index=False))
d.loc[falta_L3 & d.anio.eq(2013), "L3_ln_pib_pc"] = 0.0
requeridas_modelo = list(dict.fromkeys(xvars + ["d_ln_pib_pc"] + zvars))
if not np.isfinite(d[requeridas_modelo].to_numpy(float)).all():
    raise ValueError("Persisten valores faltantes o infinitos en la muestra del modelo.")
N = d.pais.nunique(); q = len(zvars); K = len(xvars)
if len(d) != N * 10:
    raise ValueError("Se esperan diez ecuaciones por país: 2013–2022.")
parametros = ["rho"] + X_economicas + ["shock_2020", "rebote_2021"]
d.to_csv(SALIDA / "01_muestra_transformada.csv", index=False, encoding="utf-8-sig")
print(f"N={N}; ecuaciones en diferencias={len(d)}; q={q}; K={K}.")
print("Datos 2011–2022; estimación 2013–2022. No se requiere 2010.")


N=8; ecuaciones en diferencias=80; q=7; K=6.
Datos 2011–2022; estimación 2013–2022. No se requiere 2010.


#### **Referencias MCO y FE: No resuelven automáticamente la endogeneidad**

In [6]:
formula = "ln_pib_pc ~ L1_ln_pib_pc + " + " + ".join(X_economicas)
referencias={"Pooled":smf.ols(formula+" + shock_2020 + rebote_2021",data=d).fit(),
             "FE_individual":smf.ols(formula+" + shock_2020 + rebote_2021 + C(pais)",data=d).fit(),
             "TWFE_anual":smf.ols(formula+" + C(pais) + C(anio)",data=d).fit()}
refs=pd.DataFrame({k:m.params[["L1_ln_pib_pc"]+X_economicas].to_numpy() for k,m in referencias.items()},index=parametros[:4])
print("Referencias en niveles sobre las MISMAS fechas 2013–2022:\n",refs)
print("FE dinámico: al quitar medias, el rezago transformado se correlaciona con el error transformado.")
print("El sesgo de Nickell no desaparece al aumentar N con T fijo; su orden usual es 1/T.")
print("Pooled y FE son referencias; no constituyen límites garantizados para rho en estos datos.")

Referencias en niveles sobre las MISMAS fechas 2013–2022:
                       Pooled  FE_individual  TWFE_anual
rho                 0.992935       0.825151    0.860242
inversion_fbkf      0.000112       0.001073    0.001898
apertura_comercial  0.000345       0.002207    0.001347
inflacion           0.000362      -0.002122   -0.004098
FE dinámico: al quitar medias, el rezago transformado se correlaciona con el error transformado.
El sesgo de Nickell no desaparece al aumentar N con T fijo; su orden usual es 1/T.
Pooled y FE son referencias; no constituyen límites garantizados para rho en estos datos.


#### **Matrices de difference GMM**

In [7]:
Xs=[]; Ys=[]; Zs=[]; Hs=[]; grupos=[]
for pais,g in d.groupby("pais",sort=True):
    X=g[xvars].to_numpy(float); Y=g.d_ln_pib_pc.to_numpy(float); Z=g[zvars].to_numpy(float)
    r=len(g)
    H=2*np.eye(r)-np.eye(r,k=1)-np.eye(r,k=-1)
    Xs.append(X); Ys.append(Y); Zs.append(Z); Hs.append(H); grupos.append(pais)
B=np.mean([Z.T@X for X,Z in zip(Xs,Zs)],axis=0)
a=np.mean([Z.T@Y for Y,Z in zip(Ys,Zs)],axis=0)
A1=np.mean([Z.T@H@Z for Z,H in zip(Zs,Hs)],axis=0)
assert np.linalg.matrix_rank(B)==K and np.linalg.matrix_rank(A1)==q
# H corresponde a Var(Delta u_i)/sigma² si u_it es homocedástico y no serial.
# Un peso inicial así no obliga a usar errores homocedásticos para inferencia.

#### **Un paso: coeficientes y covarianza robusta por país**

In [8]:
W1=np.linalg.solve(A1,np.eye(q))
C1=np.linalg.solve(B.T@W1@B,B.T@W1)
beta1=C1@a
scores1=np.stack([Z.T@(Y-X@beta1) for X,Y,Z in zip(Xs,Ys,Zs)])
S1=scores1.T@scores1/N
V1=C1@S1@C1.T/N
print("Rango S1:",np.linalg.matrix_rank(S1),"de",q)
print("Cada país es una unidad independiente para la covarianza robusta; no cada fila.")


Rango S1: 7 de 7
Cada país es una unidad independiente para la covarianza robusta; no cada fila.


#### **Dos pasos: nuevo peso e inferencia Windmeijer**

In [23]:
%pip install --upgrade "numpy==2.3.5" "pydynpd==0.2.2"

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.8 MB 11.7 MB/s eta 0:00:01
   -------------- ------------------------- 4.7/12.8 MB 11.4 MB/s eta 0:00:01
   ---------------------- ----------------- 7.3/12.8 MB 11.6 MB/s eta 0:00:01
   ------------------------------ --------- 9.7/12.8 MB 11.8 MB/s eta 0:00:01
   -------------------------------------- - 12.3/12.8 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 11.4 MB/s  0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
assert np.linalg.matrix_rank(S1)==q, "S1 singular: no calcular dos pasos ordinarios."
W2=np.linalg.solve(S1,np.eye(q))
C2=np.linalg.solve(B.T@W2@B,B.T@W2)
beta2=C2@a
assert metadata.version("pydynpd")=="0.2.2"
comando=("ln_pib_pc L1.ln_pib_pc inversion_fbkf apertura_comercial inflacion "
         "shock_2020 rebote_2021 | gmm(ln_pib_pc, 2:3) "
         "gmm(inversion_fbkf apertura_comercial inflacion, 2:2) "
         "iv(shock_2020 rebote_2021) | nolevel collapse")
registro=io.StringIO()
with contextlib.redirect_stdout(registro):
    ajuste=regression.abond(comando,p,["pais","anio"])
m=ajuste.models[0]
(SALIDA/"02_registro_pydynpd.txt").write_text(registro.getvalue(),encoding="utf-8")
assert np.allclose(beta1,np.asarray(m.step_results[0].beta).ravel(),atol=1e-8)
assert np.allclose(beta2,np.asarray(m.step_results[1].beta).ravel(),atol=1e-8)
V2=np.asarray(m.step_results[1].vcov)
assert np.allclose(V2,V2.T) and np.linalg.eigvalsh(V2).min()>0
print("Verificación: coeficientes manuales coinciden con pydynpd en ambos pasos.")

Verificación: coeficientes manuales coinciden con pydynpd en ambos pasos.


#### **Tablas: estimación, precisión e hipótesis individuales**

In [10]:
filas=[]
for metodo,b,V in [("Difference_1paso",beta1,V1),("Difference_2pasos_Windmeijer",beta2,V2)]:
    for j,nombre in enumerate(parametros):
        ee=np.sqrt(V[j,j]); z=b[j]/ee
        filas.append({"Modelo":metodo,"Parametro":nombre,"Beta":b[j],"EE":ee,
                      "z":z,"p_asintotico":2*norm.sf(abs(z)),
                      "IC95_inf":b[j]-norm.ppf(.975)*ee,"IC95_sup":b[j]+norm.ppf(.975)*ee})
coef=pd.DataFrame(filas)
print(coef.round(6).to_string(index=False))
print("H0 individual: coeficiente=0. Intervalos normales asintóticos, NO exactos con N=8.")
print("Windmeijer corrige covarianza de dos pasos; no elimina endogeneidad ni crea países adicionales.")
coef.to_csv(SALIDA/"03_coeficientes.csv",index=False,encoding="utf-8-sig")

                      Modelo          Parametro      Beta       EE          z  p_asintotico  IC95_inf  IC95_sup
            Difference_1paso                rho  0.947914 0.143024   6.627649      0.000000  0.667592  1.228236
            Difference_1paso     inversion_fbkf  0.001379 0.002649   0.520629      0.602625 -0.003813  0.006572
            Difference_1paso apertura_comercial  0.003123 0.000369   8.454663      0.000000  0.002399  0.003847
            Difference_1paso          inflacion -0.004566 0.002046  -2.231432      0.025653 -0.008577 -0.000555
            Difference_1paso         shock_2020 -0.083941 0.007717 -10.877472      0.000000 -0.099066 -0.068816
            Difference_1paso        rebote_2021  0.043987 0.011036   3.985745      0.000067  0.022356  0.065617
Difference_2pasos_Windmeijer                rho  0.967710 0.103237   9.373686      0.000000  0.765370  1.170051
Difference_2pasos_Windmeijer     inversion_fbkf  0.001925 0.002012   0.956466      0.338837 -0.002019  0

In [11]:
(np.exp(0.001925)-1)*100

np.float64(0.19268540019603542)

In [12]:
(np.exp(0.002907)-1)*100

np.float64(0.29112294218167634)

In [13]:
(np.exp( -0.004480)-1)*100

np.float64(-0.44699797691294396)

In [14]:
(np.exp(-0.082724)-1)*100

np.float64(-7.939480080568284)

In [15]:
(np.exp(0.047927)-1)*100

np.float64(4.9094068662079104)